# 05 — GitHub PR Template with Formulas

A pull request template where computed fields update automatically.

**What you learn:**
- `data_setter`: declare values in the reactive store
- `data_formula`: computed values that depend on other data
- Formula dependency chains resolved via topological sort
- Template sections driven by computed data

**Prerequisites:** 04_pointers

In [ ]:
from pathlib import Path
from IPython.display import Markdown

from genro_bag import Bag
from genro_builders.contrib.markdown import MarkdownBuilder
from genro_builders.manager import BuilderManager
from genro_toolbox import set_sync

set_sync()


class PullRequestTemplate(BuilderManager):
    """PR template with computed checklist and summary."""

    def on_init(self):
        self.doc = self.register_builder('doc', MarkdownBuilder)
        self.run()

    def main(self, source):
        source.data_setter('pr_title', value='^pr_title')
        source.data_setter('branch', value='^branch')
        source.data_setter('author', value='^author')
        source.data_setter('has_migration', value='^has_migration')
        source.data_setter('has_breaking_change', value='^has_breaking_change')
        source.data_setter('has_new_deps', value='^has_new_deps')
        source.data_setter('changes_summary', value='^changes_summary')

        source.data_formula(
            'pr_type',
            func=lambda pr_title: pr_title.split(':')[0] if ':' in pr_title else 'chore',
            pr_title='^pr_title',
        )

        def build_checklist(has_migration, has_breaking_change, has_new_deps):
            items = ['Tests pass locally', 'Code reviewed']
            if has_migration:
                items.append('Migration tested on staging')
            if has_breaking_change:
                items.append('CHANGELOG updated')
                items.append('Breaking change documented')
            if has_new_deps:
                items.append('New dependencies approved')
            return items

        source.data_formula(
            'checklist_items',
            func=build_checklist,
            has_migration='^has_migration',
            has_breaking_change='^has_breaking_change',
            has_new_deps='^has_new_deps',
        )

        source.data_formula(
            'checklist_text',
            func=lambda checklist_items: '\n'.join(f'- [ ] {item}' for item in checklist_items),
            checklist_items='^checklist_items',
        )

        source.h1('^pr_title')

        source.h2('Info')
        info = source.table()
        for label, pointer in [('Branch', '^branch'), ('Author', '^author'), ('Type', '^pr_type')]:
            row = info.tr()
            row.th(label)
            row.td(pointer)

        source.h2('Summary')
        source.p('^changes_summary')

        source.h2('Checklist')
        source.p('^checklist_text')

## Build and render the initial PR

In [ ]:
app = PullRequestTemplate()
app.local_store('doc').fill_from(
    Bag.from_json(Path('pr_data.json').read_text()),
)
app.doc.build()
store = app.local_store('doc')

print(f"PR type: {store['pr_type']}")
print(f"Checklist items: {store['checklist_items']}")

In [ ]:
Markdown(app.doc.render())

## Change flags — checklist updates automatically

Enable breaking change, disable migration — the checklist formula recalculates:

In [ ]:
store['has_migration'] = False
store['has_breaking_change'] = True
store['pr_title'] = 'fix!: remove deprecated auth endpoint'
store['changes_summary'] = (
    'Removes the legacy /auth/v1 endpoint. '
    'Clients must migrate to /auth/v2 with the new token format.'
)

app.doc.build()
print(f"PR type: {store['pr_type']}")
print(f"Checklist items: {store['checklist_items']}")

In [ ]:
Markdown(app.doc.render())